# Lab สัปดาห์ที่ 2: Transaction, Concurrency & Recovery
รันบน Colab ได้เลย กด Run ไล่ทีละเซลล์
(Colab ลง MySQL ผ่าน apt ได้เวอร์ชัน 8.0 แต่พฤติกรรมของ lab นี้เหมือน 8.4)

## 0. ติดตั้งและตั้งค่า

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mysql-server > /dev/null
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root'; FLUSH PRIVILEGES;"
!pip install -q pymysql

In [ ]:
import pymysql, threading, time, random, statistics

def connect(db="lab", isolation=None, autocommit=False):
    # เปิด connection ใหม่ (ตั้ง isolation level ได้)
    c = pymysql.connect(host="127.0.0.1", user="root", password="root",
                        database=db, autocommit=autocommit)
    if isolation:
        with c.cursor() as cur:
            cur.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {isolation}")
    return c

def q(c, sql, args=None):
    # รัน SQL แล้วคืนผลลัพธ์ทั้งหมด
    with c.cursor() as cur:
        cur.execute(sql, args)
        return cur.fetchall()

# สร้างฐานข้อมูลใหม่
root = connect(db=None, autocommit=True)
q(root, "DROP DATABASE IF EXISTS lab")
q(root, "CREATE DATABASE lab")
q(root, "USE lab")

q(root, "CREATE TABLE users (user_id VARCHAR(10) PRIMARY KEY, followers INT DEFAULT 0)")
q(root, """CREATE TABLE posts (
    post_id VARCHAR(10) PRIMARY KEY,
    share_count INT DEFAULT 0,
    report_count INT DEFAULT 0,
    is_flagged BOOLEAN DEFAULT FALSE)""")
q(root, """CREATE TABLE reports (
    report_id INT AUTO_INCREMENT PRIMARY KEY,
    post_id VARCHAR(10), user_id VARCHAR(10), reason VARCHAR(100),
    UNIQUE (post_id, user_id),               -- กัน report ซ้ำ
    FOREIGN KEY (post_id) REFERENCES posts(post_id),
    FOREIGN KEY (user_id) REFERENCES users(user_id))""")
q(root, "CREATE TABLE bench (id INT AUTO_INCREMENT PRIMARY KEY, val VARCHAR(20))")

def reset_data():
    # ล้างข้อมูลแล้วใส่ใหม่ (users 30 คน, post 1 โพสต์)
    q(root, "DELETE FROM reports"); q(root, "DELETE FROM posts"); q(root, "DELETE FROM users")
    with root.cursor() as cur:
        cur.executemany("INSERT INTO users(user_id) VALUES (%s)", [(f"U{i:03d}",) for i in range(1, 31)])
        cur.execute("INSERT INTO posts(post_id, share_count) VALUES ('P001', 100)")

reset_data()
print("พร้อมแล้ว")

## Lab 1: Dirty Read
T1 แก้ค่าแต่ยังไม่ commit, T2 ลองอ่านค่า เทียบ READ UNCOMMITTED กับ READ COMMITTED

In [ ]:
def dirty_read_test(level):
    t1 = connect()                  # T1 ใช้ isolation ปกติ
    t2 = connect(isolation=level)   # T2 เปลี่ยนตาม level ที่ทดสอบ
    sel = "SELECT share_count FROM posts WHERE post_id='P001'"

    q(t1, "START TRANSACTION")
    q(t1, "UPDATE posts SET share_count=777 WHERE post_id='P001'")  # ยังไม่ commit

    print(f"[{level}] T2 อ่านตอน T1 ยังไม่ commit =", q(t2, sel)[0][0])

    t1.rollback()                   # T1 ยกเลิก
    t2.rollback()                   # จบ transaction เก่าของ T2
    print(f"[{level}] T2 อ่านหลัง T1 rollback    =", q(t2, sel)[0][0])
    t1.close(); t2.close()

dirty_read_test("READ UNCOMMITTED")
print()
dirty_read_test("READ COMMITTED")

**สรุป:** READ UNCOMMITTED อ่านเห็น 777 ทั้งที่ T1 rollback ทิ้ง ค่านี้ไม่เคยมีอยู่จริง
ถ้าเอาไปใช้ เช่น แสดงยอดแชร์ คำนวณอันดับโพสต์ หรือแจ้งเตือน ก็จะได้ผลผิด
READ COMMITTED เห็นเฉพาะค่าที่ commit แล้ว (ยังคงเป็น 100)

## Lab 2: Atomic Transaction สำหรับ Report Fake News

In [ ]:
def report_fake_news(user_id, post_id, reason, threshold=3):
    c = connect()
    try:
        with c.cursor() as cur:
            # UPDATE ก่อน INSERT เพื่อล็อกแถว post ไว้ก่อน
            # (ถ้า INSERT ก่อน FK จะล็อกแบบ shared แล้วเกิด deadlock ตอนหลายคนกดพร้อมกัน)
            cur.execute("UPDATE posts SET report_count = report_count + 1 WHERE post_id=%s", (post_id,))
            cur.execute("INSERT INTO reports(post_id, user_id, reason) VALUES (%s,%s,%s)",
                        (post_id, user_id, reason))      # ซ้ำ/FK ผิด จะ error ตรงนี้
            cur.execute("UPDATE posts SET is_flagged=TRUE WHERE post_id=%s AND report_count>=%s",
                        (post_id, threshold))
        c.commit()
        return "สำเร็จ"
    except pymysql.MySQLError as e:
        c.rollback()                                      # ยกเลิกทุกอย่างใน transaction
        return f"rollback (error {e.args[0]})"
    finally:
        c.close()

reset_data()
print("1:", report_fake_news("U001", "P001", "ข่าวปลอม"))
print("2 (ซ้ำ):", report_fake_news("U001", "P001", "ข่าวปลอม"))
print("3 (post ไม่มี):", report_fake_news("U002", "P999", "ข่าวปลอม"))
print(q(root, "SELECT post_id, report_count, is_flagged FROM posts"))
print("จำนวนแถวใน reports:", q(root, "SELECT COUNT(*) FROM reports")[0][0])

duplicate (1062) กับ FK error (1452) ต้อง rollback ทั้งชุด ดังนั้น `report_count` ต้องเป็น 1 ไม่ใช่ 2 หรือ 3

### ทดสอบ concurrent: 30 คน report โพสต์เดียวกันพร้อมกัน

In [ ]:
def run_threads(fn, args_list):
    ts = [threading.Thread(target=fn, args=a) for a in args_list]
    for t in ts: t.start()
    for t in ts: t.join()

# แบบถูก: UPDATE report_count = report_count + 1 ให้ DB นับเอง (atomic)
reset_data()
run_threads(report_fake_news, [(f"U{i:03d}", "P001", "ปลอม", 10) for i in range(1, 31)])
print("แบบ atomic UPDATE :", q(root, "SELECT report_count, is_flagged FROM posts"), "(ควรได้ 30)")

# แบบผิด: อ่านค่ามาบวกใน Python แล้วเขียนกลับ -> lost update
def bad_report(uid):
    c = connect()
    n = q(c, "SELECT report_count FROM posts WHERE post_id='P001'")[0][0]
    time.sleep(0.01)                                   # จำลองว่าประมวลผลนิดนึง
    q(c, "UPDATE posts SET report_count=%s WHERE post_id='P001'", (n + 1,))
    c.commit(); c.close()

reset_data()
run_threads(bad_report, [(f"U{i:03d}",) for i in range(1, 31)])
print("แบบอ่านแล้วเขียน   :", q(root, "SELECT report_count FROM posts"), "(น้อยกว่า 30 = ข้อมูลหาย)")

**สรุป:** ควรให้ DB บวกเอง (`SET x = x + 1`) หรือถ้าต้องอ่านก่อนใช้ `SELECT ... FOR UPDATE` เพื่อล็อกแถว
ไม่ควรอ่านค่ามาคำนวณนอก DB แล้วเขียนทับ

## Lab 3: Benchmark Autocommit vs Explicit Transaction

In [ ]:
print("innodb_flush_log_at_trx_commit =", q(root, "SELECT @@innodb_flush_log_at_trx_commit")[0][0])

def bench(n, autocommit):
    c = connect(autocommit=autocommit)
    cur = c.cursor()
    cur.execute("TRUNCATE bench")
    t = time.perf_counter()
    for i in range(n):
        cur.execute("INSERT INTO bench(val) VALUES (%s)", (f"row{i}",))
    if not autocommit:
        c.commit()                     # commit ครั้งเดียวตอนท้าย
    dt = time.perf_counter() - t
    c.close()
    return dt

bench(100, True); bench(100, False)    # warm up

ROUNDS = 5
for n in (100, 1000):
    a = [bench(n, True)  for _ in range(ROUNDS)]
    b = [bench(n, False) for _ in range(ROUNDS)]
    ma, mb = statistics.median(a), statistics.median(b)
    print(f"INSERT {n:>4} แถว | autocommit: {ma:.3f}s | transaction เดียว: {mb:.3f}s | ต่างกัน {ma/mb:.1f} เท่า")

**อธิบาย:** autocommit จะ commit ทุก statement และทุกครั้งที่ commit InnoDB ต้อง flush redo log ลงดิสก์ (fsync)
เมื่อ `innodb_flush_log_at_trx_commit=1` ซึ่งเป็นค่า default เลยช้า การรวมเป็น transaction เดียวจึงเร็วกว่ามาก
แต่ transaction ที่ใหญ่เกินไปก็มีข้อเสีย: undo log โต, ล็อกแถวไว้นาน (คนอื่นต้องรอ), ถ้าล้มเหลวต้อง rollback/recovery นาน
ทางที่ดีคือแบ่ง commit เป็นก้อน เช่นทุก 500-1000 แถว

## Lab 4: Deadlock Detection และ Retry
T1 อัปเดต U001 -> U002, T2 อัปเดต U002 -> U001 (สลับลำดับกัน)

In [ ]:
reset_data()
upd = "UPDATE users SET followers=followers+1 WHERE user_id=%s"
results = {}

def worker(name, first, second):
    c = connect()
    try:
        q(c, "START TRANSACTION")
        q(c, upd, (first,))
        time.sleep(1)                  # รอให้อีกฝั่งล็อกแถวของมันก่อน
        q(c, upd, (second,))
        c.commit()
        results[name] = "commit สำเร็จ"
    except pymysql.MySQLError as e:
        c.rollback()
        results[name] = f"ถูก rollback (error {e.args[0]})"   # 1213 = deadlock
    finally:
        c.close()

run_threads(worker, [("T1", "U001", "U002"), ("T2", "U002", "U001")])
print(results)

In [ ]:
# ดูรายละเอียด deadlock ล่าสุด
status = q(root, "SHOW ENGINE INNODB STATUS")[0][2]
s = status.find("LATEST DETECTED DEADLOCK")
e = status.find("TRANSACTIONS", s + 30)
print(status[s:e])

### Retry ด้วย exponential backoff + jitter

In [ ]:
def run_with_retry(fn, max_retry=5, base=0.1):
    for i in range(max_retry + 1):
        try:
            return fn()
        except pymysql.err.OperationalError as e:
            # 1213 = deadlock, 1205 = lock wait timeout เท่านั้นที่ลองใหม่
            if e.args[0] not in (1213, 1205) or i == max_retry:
                raise
            wait = base * 2**i + random.uniform(0, base)   # รอนานขึ้นเรื่อยๆ + สุ่มกันชนกันอีก
            print(f"  deadlock ครั้งที่ {i+1}, รอ {wait:.2f}s แล้วลองใหม่ทั้ง transaction")
            time.sleep(wait)

def tx(first, second):
    c = connect()
    try:
        q(c, upd, (first,))
        time.sleep(0.5)
        q(c, upd, (second,))
        c.commit()
    except Exception:
        c.rollback()
        raise
    finally:
        c.close()

reset_data()
run_threads(lambda a, b: run_with_retry(lambda: tx(a, b)),
            [("U001", "U002"), ("U002", "U001")])
print(q(root, "SELECT user_id, followers FROM users WHERE user_id IN ('U001','U002')"),
      "(ทั้งสองคนควรได้ 2 = ทั้งสอง transaction สำเร็จ)")

**สรุป:** deadlock ป้องกันไม่ได้ 100% วิธีลดคือให้ทุก transaction ล็อกแถวตามลำดับเดียวกัน (เช่นเรียง user_id)
และต้องมี retry ทั้ง transaction (ไม่ใช่แค่ statement ที่พัง) เพราะ InnoDB rollback ทั้ง transaction ให้แล้ว